In [ ]:
# A = [[1, 2], [3, 4]]
# B = [[1, 2, 3], [4, 5, 6]]
import math
import numpy as np
from torchvision.datasets import MNIST
from tqdm.auto import tqdm

def matrix_mult(M, N):
    m, n, r = len(M), len(M[0]), len(N[0])
    assert n == len(N), f"{m}x{n} and {len(N)}x{r} matrices sizes incompatible"

    result = [[0 for _ in range(r)] for _ in range(m)]

    for i in range(m):
        for j in range(r):
            sum = 0
            for k in range(n):
                sum += M[i][k] * N[k][j]
            result[i][j] = sum
    return result

def matrix_scalar_mult(c, M):
    m, n = len(M), len(M[0])
    scaled = [[c * M[i][j] for j in range(n)] for i in range(m)]

    return scaled

def matrix_add(M, N):
    m, n = len(M), len(M[0])
    assert m == len(N) and n == len(N[0]), f"{m}x{n} and {len(N)}x{len(N[0])} matrices sizes incompatible"

    result = [[M[i][j] + N[i][j] for j in range(n)] for i in range(m)]
    return result

def matrix_sub(M, N):
    m, n = len(M), len(M[0])
    assert m == len(N) and n == len(N[0]), f"{m}x{n} and {len(N)}x{len(N[0])} matrices sizes incompatible"

    result = [[M[i][j] - N[i][j] for j in range(n)] for i in range(m)]
    return result

def hadamard(M, N):
    m, n = len(M), len(M[0])
    assert m == len(N) and n == len(N[0]), f"{m}x{n} and {len(N)}x{len(N[0])} matrices sizes incompatible"

    result = [[M[i][j] * N[i][j] for j in range(n)] for i in range(m)]
    return result


def transpose(M):
    m, n = len(M), len(M[0])
    # print(f"{m} x {n}")
    transposed = [[0.0 for _ in range(m)] for _ in range(n)]

    for i in range(m):
        for j in range(n):
            transposed[j][i] = M[i][j]

    return transposed


def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def dsigmoid(x):
    return sigmoid(x) * (1-sigmoid(x))

def vector_dsigmoid(z):
    return [[dsigmoid(z[i][0])] for i in range(len(z))]


def vector_sigmoid(z):
    return [[sigmoid(z[i][0])] for i in range(len(z))]


# quadratic loss
def C_x(y, a):
    loss = 0.0
    for i in range(len(y)):
        loss += (y[i][0] - a[i][0]) ** 2
    loss /= 2
    return loss


# cross entropy loss
# def C_x(y, a):
#     loss = 0.0
#     for i in range(len(y)):
#         loss -= y[i][0]*math.log(a[i][0]) + (1 - y[i][0])*math.log((1 - a[i][0]))
#     return loss



def acc(y, a):
    return np.argmax(y) == np.argmax(a)

class Layer:
    def __init__(self, input_neurons, output_neurons):
        self.j = output_neurons
        self.k = input_neurons

        self.W = np.random.normal(scale = 1/math.sqrt(self.j), size = (self.j, self.k)).tolist() # mean = 0, sd = 1.0
        self.b = np.random.normal(size = (self.j, 1)).tolist()

        self.clear_grad()

    def __call__(self, prev_activations):
        return self.forward(prev_activations)

    def forward(self, prev_activations):
        z = matrix_add(matrix_mult(self.W, prev_activations), self.b)
        return vector_sigmoid(z), z

    def clear_grad(self):
        self.W_grad = [[0.0 for _ in range(self.k)] for _ in range(self.j)]
        self.b_grad = [[0.0] for _ in range(self.j)]

class Optimizer:
    def __init__(self, lr, batch_size):
        self.lr = lr
        self.batch_size = batch_size

    def __call__(self, layer): # update the parameters
        # print(f"{len(layer.W)}x{len(layer.W[0])}, {len(layer.W_grad)}x{len(layer.W_grad[0])}")
        layer.W = matrix_sub(layer.W, matrix_scalar_mult(self.lr/self.batch_size, layer.W_grad))
        layer.b = matrix_sub(layer.b, matrix_scalar_mult(self.lr/self.batch_size, layer.b_grad))





class MLP:
    def __init__(self, hidden_neurons):
        self.layers = [
            Layer(784, hidden_neurons),
            Layer(hidden_neurons, 10),
        ]
        self.clear_a_and_z()
        self.optim = Optimizer(1e-3, 64)



    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        output = x

        for layer in self.layers: 
            output, weighted_input = layer(output)
            self.activations.append(output)
            self.weighted_inputs.append(weighted_input)

        return output

    def backward(self, x, y):
        L = len(self.layers)
        delta_l_s = [0.0 for _ in range(L)]

        for l in range(L-1, -1, -1):
            a_l = self.activations[l]
            z_l = self.weighted_inputs[l]

            # print(l)
            
            if (l == L - 1): 
                # print(matrix_sub(a_l, y))
                delta_l = hadamard(matrix_sub(a_l, y), vector_dsigmoid(z_l)) # BP1
            else:
                # print(matrix_mult(transpose(self.layers[l+1].W), delta_l_s[l+1]))
                delta_l = hadamard(matrix_mult(transpose(self.layers[l+1].W), delta_l_s[l+1]), vector_dsigmoid(z_l)) # BP2

            delta_l_s[l] = delta_l

            # BP3
            if (l == 0):
                W_grad = matrix_mult(delta_l, transpose(x))
            else:
                W_grad = matrix_mult(delta_l, transpose(self.activations[l-1]))


            self.layers[l].W_grad = matrix_add(self.layers[l].W_grad, W_grad)
            self.layers[l].b_grad = matrix_add(self.layers[l].b_grad, delta_l) # BP4

            # print(self.layers[l].W_grad)

            # print(f"l: {l} | grad: {self.layers[l].W_grad}")

        self.clear_a_and_z()

        


    def step(self): # update parameters 
        for layer in self.layers:
            self.optim(layer)

    def clear_a_and_z(self):
        self.activations = []
        self.weighted_inputs = []


    def clear_grad(self):
        for layer in self.layers:
            layer.clear_grad()

    def set_optim(self, optim):
        self.optim = optim
        





         


In [2]:
train = MNIST(root="data", train=True, download=True)
test = MNIST(root="data", train=False, download=True)

x_train, y_train = [], []
x_test, y_test = [], []

for image, label in train:
    x = [[pixel / 255.0] for pixel in image.getdata()]
    y = [[0.0] for _ in range(10)]
    y[label] = [1.0]

    x_train.append(x)
    y_train.append(y)


for image, label in test:
    x = [[pixel / 255.0] for pixel in image.getdata()]
    y = [[0.0] for _ in range(10)]
    y[label] = [1.0]

    x_test.append(x)
    y_test.append(y)

x_val = x_train[40000:]
y_val = y_train[40000:]

x_train = x_train[:40000]
y_train = y_train[:40000]

In [10]:
np.random.seed(42)

mlp = MLP(hidden_neurons=30)
batch_size = 128
epochs = 50
lr = 3.0

optim = Optimizer(lr=lr, batch_size=batch_size)
mlp.set_optim(optim=optim)

for i in range(epochs):
    num_train_samples = len(x_train)
    train_loss = 0.0

    combined = list(zip(x_train, y_train))
    np.random.shuffle(combined)
    x_train, y_train = zip(*combined)

    pbar = tqdm(total=num_train_samples, unit="samples")

    for batch_index in range(0, num_train_samples, batch_size):
        for offset in range(batch_size):
            s = batch_index + offset
            if (s >= num_train_samples): break

            x, y = x_train[s], y_train[s]
            preds = mlp(x)
            mlp.backward(x, y) 
            # sys.exit()
            loss = C_x(y, preds)
            train_loss += loss

            # if s % 1024 == 0: print(loss)
            pbar.update(1)
            # if (s % 1000 == 0): print(f"{s}/{num_samples}")

        mlp.step()
        mlp.clear_grad()


    pbar.close()

    train_loss /= num_train_samples

    correct = 0.0
    num_val_samples = len(x_val)

    for s in range(num_val_samples):
        x, y = x_val[s], y_val[s]

        preds = mlp(x)
        correct += acc(y, preds)    

    val_acc = 100 * correct / num_val_samples   
        
        

    
    print(f"Epoch: {i+1} | Loss: {train_loss:.4f} | Acc: {val_acc:.2f}%")  




  0%|          | 0/40000 [00:00<?, ?samples/s]

Epoch: 1 | Loss: 0.1904 | Acc: 89.64%


  0%|          | 0/40000 [00:00<?, ?samples/s]

Epoch: 2 | Loss: 0.0878 | Acc: 90.97%


  0%|          | 0/40000 [00:00<?, ?samples/s]

...

Epoch: 48 | Loss: 0.0236 | Acc: 95.25%


  0%|          | 0/40000 [00:00<?, ?samples/s]

Epoch: 49 | Loss: 0.0234 | Acc: 95.25%


  0%|          | 0/40000 [00:00<?, ?samples/s]

Epoch: 50 | Loss: 0.0232 | Acc: 95.24%


In [12]:
correct = 0.0
num_test_samples = len(x_test)


for s in range(num_test_samples):
        x, y = x_test[s], y_test[s]

        preds = mlp(x)
        correct += acc(y, preds)
        
test_acc = 100 * correct / num_test_samples

print(f"Test Acc: {test_acc:.2f}%")



Test Acc: 95.62%
